# Resume Candidate Screening System

This notebook builds an ML-powered resume screening system that ranks candidates based on job description matching using TF-IDF and cosine similarity.

## Features
- Synthetic dataset: 50 candidate resumes
- spaCy for skill extraction from resumes
- TF-IDF vectorization for job description matching
- Cosine similarity ranking
- Top 5 candidates with skill gap analysis
- Visualizations and metrics
- Live demo with 3 new resumes

In [ ]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print('All imports successful!')

In [ ]:
# Cell 2: Load spaCy model
nlp = spacy.load('en_core_web_sm')
print('spaCy model loaded successfully!')

In [ ]:
# Cell 3: Synthetic Data - 50 Resumes
np.random.seed(42)

skills_pool = [
    'Python', 'Java', 'C++', 'JavaScript', 'SQL', 'R', 'Machine Learning',
    'Deep Learning', 'TensorFlow', 'PyTorch', 'Scikit-learn', 'Pandas',
    'NumPy', 'Data Visualization', 'Tableau', 'Power BI', 'Statistics',
    'Data Mining', 'NLP', 'Computer Vision', 'AWS', 'Azure', 'Docker',
    'Kubernetes', 'Git', 'Agile', 'Communication', 'Problem Solving'
]

experience_levels = ['Fresher', '1-2 years', '2-4 years', '4-6 years', '6+ years']
degrees = ['B.Tech', 'M.Tech', 'B.Sc', 'M.Sc', 'B.E', 'MCA']

names = [f'Candidate_{i}' for i in range(1, 51)]

resumes = []
for i in range(50):
    num_skills = np.random.randint(4, 12)
    candidate_skills = list(np.random.choice(skills_pool, num_skills, replace=False))
    exp = np.random.choice(experience_levels)
    degree = np.random.choice(degrees)
    resume_text = f'Name: {names[i]}. {degree} graduate with {exp} experience. Skills include ' + ', '.join(candidate_skills) + '.'
    resumes.append({'id': i+1, 'name': names[i], 'text': resume_text, 'skills': candidate_skills, 'experience': exp, 'degree': degree})

df = pd.DataFrame(resumes)
print(f'Generated {len(df)} resumes')

In [ ]:
# Cell 4: Job Description
job_description = '''
Data Scientist - Job Description
We are looking for a Data Scientist with strong Python, Machine Learning, and SQL skills.
Required skills: Python, Machine Learning, Deep Learning, TensorFlow, PyTorch, Scikit-learn,
Pandas, NumPy, Data Visualization, Statistics, Data Mining, NLP, SQL.
Experience: 2-4 years preferred. Degree: B.Tech or M.Tech in Computer Science.
'''

# Required skills for the job
required_skills = ['Python', 'Machine Learning', 'Deep Learning', 'TensorFlow', 'PyTorch',
                   'Scikit-learn', 'Pandas', 'NumPy', 'Data Visualization', 'Statistics',
                   'Data Mining', 'NLP', 'SQL']

print('Job description defined with', len(required_skills), 'required skills')

In [ ]:
# Cell 5: TF-IDF Vectorization
# Prepare corpus
corpus = [resume['text'] for resume in resumes] + [job_description]

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
tfidf_matrix = vectorizer.fit_transform(corpus)

# Get vectors for each resume and the job description
resume_vectors = tfidf_matrix[:-1]
job_vector = tfidf_matrix[-1:]

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')

In [ ]:
# Cell 6: Cosine Similarity Ranking
# Calculate cosine similarity between each resume and job description
similarities = cosine_similarity(resume_vectors, job_vector).flatten()

# Add similarity scores to dataframe
df['similarity_score'] = similarities
df['match_percentage'] = (similarities * 100).round(2)

# Rank candidates
df = df.sort_values('similarity_score', ascending=False).reset_index(drop=True)
df['rank'] = range(1, len(df)+1)

print('\nTop 10 Candidates:')
print(df[['rank', 'name', 'match_percentage', 'experience']].head(10).to_string(index=False))

In [ ]:
# Cell 7: Visualization - Candidate Ranking Bar Chart
plt.figure(figsize=(12, 6))
top10 = df.head(10)
colors = ['#2ecc71' if x >= 70 else '#f39c12' if x >= 50 else '#e74c3c' for x in top10['match_percentage']]
bars = plt.bar(range(1, 11), top10['match_percentage'], color=colors)
plt.xlabel('Candidate Rank')
plt.ylabel('Match Percentage')
plt.title('Top 10 Candidates - Job Description Match')
plt.xticks(range(1, 11), top10['name'], rotation=45, ha='right')
plt.axhline(y=70, color='green', linestyle='--', alpha=0.7, label='Strong Match (>70%)')
plt.axhline(y=50, color='orange', linestyle='--', alpha=0.7, label='Moderate Match (>50%)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Skill Gap Analysis for Top 5
top5 = df.head(5)

print('\n' + '='*60)
print('TOP 5 CANDIDATES - SKILL GAP ANALYSIS')
print('='*60)

for idx, row in top5.iterrows():
    print(f'\nRank #{row["rank"]}: {row["name"]} - {row["match_percentage"]}% Match')
    print(f'Experience: {row["experience"]}, Degree: {row["degree"]}')
    
    matched = [s for s in row['skills'] if s in required_skills]
    missing = [s for s in required_skills if s not in row['skills']]
    
    print(f'Matched Skills ({len(matched)}): ' + ', '.join(matched))
    print(f'Missing Skills ({len(missing)}): ' + ', '.join(missing) if missing else 'None!')

In [ ]:
# Cell 9: Metrics - Match Distribution & Top-3 Accuracy
top3_strong = len(df[df['rank'] <= 3][df['match_percentage'] > 70]) / 3 * 100

all_skills_covered = set()
for skills in df['skills']:
    all_skills_covered.update(skills)

skills_coverage = len([s for s in required_skills if s in all_skills_covered]) / len(required_skills) * 100

print('='*60)
print('PERFORMANCE METRICS')
print('='*60)
print(f'Total Resumes Screened: {len(df)}')
print(f'Average Match Score: {df["match_percentage"].mean():.2f}%')
print(f'Best Match Score: {df["match_percentage"].max():.2f}%')
print(f'Top-3 Strong Match Rate: {top3_strong:.2f}%')
print(f'Required Skills Coverage: {skills_coverage:.2f}%')
print(f'Candidates with >70% Match: {len(df[df["match_percentage"] > 70])}')
print(f'Candidates with >50% Match: {len(df[df["match_percentage"] > 50])}')

In [ ]:
# Cell 10: Skill Match Heatmap for Top 5
top5_idx = df.head(5).index
skill_matrix = pd.DataFrame(index=top5_idx, columns=required_skills)

for idx in top5_idx:
    for skill in required_skills:
        skill_matrix.loc[idx, skill] = 1 if skill in df.loc[idx, 'skills'] else 0

plt.figure(figsize=(12, 6))
sns.heatmap(skill_matrix.T, annot=True, cmap='RdYlGn', fmt='.0f',
            cbar_kws={'label': '1=Has Skill, 0=Missing'},
            yticklabels=required_skills)
plt.title('Skill Match Heatmap - Top 5 Candidates')
plt.xlabel('Candidate Rank')
plt.xticks(range(5), [f'#{i+1}' for i in range(5)])
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Live Demo - Screen 3 New Resumes
demo_resumes = [
    {'id': 51, 'name': 'Demo_Candidate_A', 'text': 'Name: Demo_Candidate_A. M.Tech graduate with 3 years experience in Machine Learning and Data Science. Skills include Python, Machine Learning, TensorFlow, Scikit-learn, Pandas, NumPy, SQL, Data Visualization, NLP.', 'experience': '2-4 years'},
    {'id': 52, 'name': 'Demo_Candidate_B', 'text': 'Name: Demo_Candidate_B. B.Tech graduate with 1 year experience in Software Development. Skills include Java, JavaScript, SQL, Git, Agile, Communication.', 'experience': '1-2 years'},
    {'id': 53, 'name': 'Demo_Candidate_C', 'text': 'Name: Demo_Candidate_C. M.Sc graduate with 5 years experience in Data Analytics. Skills include Python, R, Deep Learning, PyTorch, Statistics, Data Mining, Tableau, Power BI, AWS.', 'experience': '4-6 years'}
]

demo_corpus = [r['text'] for r in demo_resumes]
demo_vectors = vectorizer.transform(demo_corpus)
demo_sims = cosine_similarity(demo_vectors, job_vector).flatten()

print('='*60)
print('LIVE DEMO: Screening 3 New Candidates')
print('='*60)

for i, (res, sim) in enumerate(zip(demo_resumes, demo_sims), 1):
    match_pct = sim * 100
    status = 'RECOMMENDED' if match_pct > 60 else 'CONSIDER' if match_pct > 40 else 'NOT SUITABLE'
    print(f'\n{i}. {res["name"]} - {match_pct:.2f}% Match - [{status}]')
    print(f'   Experience: {res["experience"]}')

In [ ]:
# Cell 12: Business Insights Summary
print('='*60)
print('BUSINESS INSIGHTS SUMMARY')
print('='*60)

print('\n1. SCREENING EFFICIENCY:')
print(f'   - System screened {len(df)} candidates in seconds')
print(f'   - Top 5 candidates identified with skill gap analysis')
print(f'   - Top-3 accuracy rate: {top3_strong:.2f}%')

print('\n2. RECOMMENDATIONS:')
if len(df[df["match_percentage"] > 70]) >= 3:
    print('   - Strong candidate pool available. Recommend proceeding with top 5.')
else:
    print('   - Consider expanding search or adjusting requirements.')

print('\n3. SKILL GAPS ACROSS CANDIDATES:')
skill_gaps = {}
for skill in required_skills:
    coverage = len([s for s in df['skills'] if skill in s]) / len(df) * 100
    skill_gaps[skill] = coverage
for skill, cov in sorted(skill_gaps.items(), key=lambda x: x[1]):
    if cov < 60:
        print(f'   - {skill}: Only {cov:.1f}% of candidates have this skill')

print('\n' + '='*60)
print('END OF NOTEBOOK')
print('='*60)